# Mobile Games A/B Testing: Cookie Cats Gate Placement Analysis
**Experiment:** Progression Gate at Level 30 (Control) vs. Level 40 (Treatment)  
**Dataset:** Cookie Cats Player Retention (90,189 players)  
**Author:** Sanjeevin  

---

## 1. Executive Summary & Experiment Context
Cookie Cats is a classic mobile puzzle game developed by Tactile Entertainment. As players progress through the game, they encounter "gates" that force them to either wait a non-trivial amount of time or make an in-app purchase to proceed.

* **The Product Decision:** Should the first gate be placed at **Level 30** (early gate) or moved back to **Level 40** (delayed gate)?
* **Product Hypothesis:** Delaying the gate to Level 40 allows players to experience more gameplay without artificial disruption, potentially improving long-term player retention and engagement.
* **Counter-Hypothesis (Hedonic Adaptation):** Forcing players to take a mandatory break earlier (Gate 30) prevents player burnout, creates anticipation, and establishes a sustainable playing habit.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Configure styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 130, "axes.titlesize": 12, "axes.labelsize": 10})

# Determine working paths robustly
BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else os.getcwd()
ROOT_DIR = os.path.abspath(os.path.join(BASE_DIR, '..')) if os.path.basename(BASE_DIR) == 'notebooks' else BASE_DIR

DATA_PATH = os.path.join(ROOT_DIR, 'data', 'raw', 'cookie_cats.csv')
OUTPUT_DIR = os.path.join(ROOT_DIR, 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 2. Data Ingestion & Data Hygiene Audit

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print(f"Total raw players recorded: {len(df_raw):,}")
print(df_raw.info())
print("\nFirst 5 records:")
display(df_raw.head()) if 'display' in dir() else print(df_raw.head())

### Outlier Audit on `sum_gamerounds`
Before running any statistical inference, we audit the distribution of game rounds played in the first 14 days.

In [ ]:
print("Summary statistics of sum_gamerounds:")
print(df_raw['sum_gamerounds'].describe())

# Detect extreme outliers
max_gamerounds = df_raw['sum_gamerounds'].max()
print(f"\nMax game rounds played: {max_gamerounds:,}")
outlier_user = df_raw[df_raw['sum_gamerounds'] == max_gamerounds]
print("Outlier row:")
print(outlier_user)

# Filter out the single impossible bot/counter outlier (49,854 rounds = 2.5 rounds/min continuous for 14 days)
df = df_raw[df_raw['sum_gamerounds'] < 40000].copy()
print(f"\nClean sample size: {len(df):,} players (1 outlier removed).")
print(f"Max rounds in clean sample: {df['sum_gamerounds'].max():,}")

## 3. Experiment Design & Metric Framework

* **Primary Metric:** **7-Day Retention (`retention_7`)**  
  *Definition:* Boolean flag indicating whether the player returned to the game 7 days after installation.  
  *Why:* Measures habit formation and long-term player lifetime value (LTV).
* **Secondary Metric:** **1-Day Retention (`retention_1`)**  
  *Definition:* Boolean flag indicating whether the player returned 1 day after installation.  
  *Why:* Early indicator of initial onboarding engagement.
* **Guardrail Metric:** **Game Rounds Played (`sum_gamerounds`)**  
  *Definition:* Total rounds completed in the first 14 days.  
  *Why:* Ensures that gate changes do not suppress total player activity.

## 4. Sample Ratio Mismatch (SRM) Diagnostics
An A/B test is invalid if the randomization mechanism was corrupted, creating an imbalance between variants.  
Expected split: 50% Control (`gate_30`) vs. 50% Treatment (`gate_40`).

In [ ]:
observed_counts = df['version'].value_counts()
n_total = len(df)
expected_counts = [n_total / 2, n_total / 2]

chi2_stat, p_srm = stats.chisquare(observed_counts, expected_counts)

print("--- Sample Ratio Mismatch (SRM) Test ---")
print(f"Observed - gate_40: {observed_counts['gate_40']:,} ({observed_counts['gate_40']/n_total*100:.2f}%)")
print(f"Observed - gate_30: {observed_counts['gate_30']:,} ({observed_counts['gate_30']/n_total*100:.2f}%)")
print(f"Chi-Square Statistic: {chi2_stat:.4f}")
print(f"p-value: {p_srm:.4f}")

if p_srm < 0.01:
    print("\nRESULT: Statistically significant SRM detected (p < 0.01).")
    print("Interpretation: While a 50.44% vs 49.56% split is practically minor, in production we investigate variant assignment latency.")
else:
    print("\nRESULT: No evidence of SRM. Randomization split is valid.")

## 5. Statistical Power & Minimum Detectable Effect (MDE)
We evaluate whether the sample size ($N \approx 90,000$) provided adequate statistical power ($1 - \beta \ge 0.80$) to detect meaningful shifts in 7-day retention.

In [ ]:
p_base = df[df['version'] == 'gate_30']['retention_7'].mean()
n_ctrl = len(df[df['version'] == 'gate_30'])
n_treat = len(df[df['version'] == 'gate_40'])

# Standard error of difference under pooled null
p_pool_null = (df['retention_7'].sum()) / n_total
se_diff = np.sqrt(p_pool_null * (1 - p_pool_null) * (1/n_ctrl + 1/n_treat))

# Z critical values
z_alpha = stats.norm.ppf(1 - 0.05 / 2) # 1.96 for alpha = 0.05
z_beta_80 = stats.norm.ppf(0.80)      # 0.8416 for 80% power

mde_abs = (z_alpha + z_beta_80) * se_diff
mde_rel = (mde_abs / p_base) * 100

print("--- Power & Sample Size Audit ---")
print(f"Baseline 7-Day Retention (Gate 30): {p_base*100:.2f}%")
print(f"Sample Size: Control = {n_ctrl:,}, Treatment = {n_treat:,}")
print(f"Minimum Detectable Effect (Absolute): +/- {mde_abs*100:.3f}%")
print(f"Minimum Detectable Effect (Relative): +/- {mde_rel:.2f}%")
print("Interpretation: The test had 80% power to detect any true retention shift larger than 0.726% absolute (3.82% relative).")

## 6. Hypothesis Testing: Two-Proportion Z-Tests

In [ ]:
def run_ztest(metric_name):
    agg = df.groupby('version')[metric_name].agg(['count', 'sum', 'mean'])
    c30, n30 = agg.loc['gate_30', 'sum'], agg.loc['gate_30', 'count']
    c40, n40 = agg.loc['gate_40', 'sum'], agg.loc['gate_40', 'count']
    
    p30, p40 = c30 / n30, c40 / n40
    diff = p40 - p30
    rel_diff = (diff / p30) * 100
    
    p_pool = (c30 + c40) / (n30 + n40)
    se = np.sqrt(p_pool * (1 - p_pool) * (1/n30 + 1/n40))
    z_stat = diff / se
    p_val = 2 * (1 - stats.norm.cdf(abs(z_stat)))
    
    ci_low = diff - z_alpha * se
    ci_high = diff + z_alpha * se
    
    return {
        'Metric': metric_name,
        'Gate 30': p30 * 100,
        'Gate 40': p40 * 100,
        'Absolute Diff': diff * 100,
        'Relative Diff (%)': rel_diff,
        'Z-Statistic': z_stat,
        'p-value': p_val,
        '95% CI Lower': ci_low * 100,
        '95% CI Upper': ci_high * 100
    }

res_1d = run_ztest('retention_1')
res_7d = run_ztest('retention_7')

df_results = pd.DataFrame([res_1d, res_7d])
display(df_results) if 'display' in dir() else print(df_results.to_string())

### Z-Test Statistical Insights:
1. **1-Day Retention:** Dropped from **44.82%** (Gate 30) to **44.23%** (Gate 40) ($-0.59\%$ absolute, $-1.32\%$ relative).  
   * $z = -1.787$, $p = 0.0739$. Not statistically significant at $\alpha = 0.05$.
2. **7-Day Retention (Primary Metric):** Dropped from **19.02%** (Gate 30) to **18.20%** (Gate 40) ($-0.82\%$ absolute, $-4.30\%$ relative).  
   * $z = -3.157$, **$p = 0.00159$ ($p < 0.01$)**. Highly statistically significant.

## 7. Engagement & Guardrail Analysis: Game Rounds Played

In [ ]:
rounds_summary = df.groupby('version')['sum_gamerounds'].agg(['count', 'mean', 'median', 'std'])
display(rounds_summary) if 'display' in dir() else print(rounds_summary)

rounds_30 = df[df['version'] == 'gate_30']['sum_gamerounds']
rounds_40 = df[df['version'] == 'gate_40']['sum_gamerounds']

# Non-parametric Mann-Whitney U test due to heavy skew
u_stat, p_u = stats.mannwhitneyu(rounds_30, rounds_40, alternative='two-sided')
# Parametric Welch's t-test
t_stat, p_t = stats.ttest_ind(rounds_30, rounds_40, equal_var=False)

print(f"\nMann-Whitney U Test: U = {u_stat:,.1f}, p-value = {p_u:.4f}")
print(f"Welch's t-test: t = {t_stat:.4f}, p-value = {p_t:.4f}")

## 8. Non-Parametric Bootstrap Confidence Intervals (10,000 Iterations)
To avoid relying solely on asymptotic normality assumptions, we execute a 10,000-iteration bootstrap simulation.

In [ ]:
np.random.seed(42)
g30 = df[df['version'] == 'gate_30']
g40 = df[df['version'] == 'gate_40']

n30 = len(g30)
n40 = len(g40)

# High-speed exact bootstrap simulation
p30_1_boot = np.random.binomial(n30, g30['retention_1'].mean(), 10000) / n30
p40_1_boot = np.random.binomial(n40, g40['retention_1'].mean(), 10000) / n40
boot_diff_1d = (p40_1_boot - p30_1_boot) * 100

p30_7_boot = np.random.binomial(n30, g30['retention_7'].mean(), 10000) / n30
p40_7_boot = np.random.binomial(n40, g40['retention_7'].mean(), 10000) / n40
boot_diff_7d = (p40_7_boot - p30_7_boot) * 100

# Percentile confidence intervals
ci_1d = np.percentile(boot_diff_1d, [2.5, 97.5])
ci_7d = np.percentile(boot_diff_7d, [2.5, 97.5])

prob_gate30_better_1d = (boot_diff_1d < 0).mean() * 100
prob_gate30_better_7d = (boot_diff_7d < 0).mean() * 100

print("--- Bootstrap Results (10,000 Iterations) ---")
print(f"1-Day Retention Diff (Gate 40 - Gate 30): Mean = {boot_diff_1d.mean():.3f}%, 95% CI: [{ci_1d[0]:.3f}%, {ci_1d[1]:.3f}%]")
print(f"Probability Gate 30 is superior for 1-Day Retention: {prob_gate30_better_1d:.2f}%\n")
print(f"7-Day Retention Diff (Gate 40 - Gate 30): Mean = {boot_diff_7d.mean():.3f}%, 95% CI: [{ci_7d[0]:.3f}%, {ci_7d[1]:.3f}%]")
print(f"Probability Gate 30 is superior for 7-Day Retention: {prob_gate30_better_7d:.2f}%")

### Visualization of Bootstrap Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Plot 1-Day Retention Diff
sns.histplot(boot_diff_1d, kde=True, ax=axes[0], color="#4C72B0", stat="density", bins=40)
axes[0].axvline(0, color="red", linestyle="--", linewidth=1.5, label="No Difference (0%)")
axes[0].axvline(ci_1d[0], color="black", linestyle=":", label=f"95% CI: [{ci_1d[0]:.2f}%, {ci_1d[1]:.2f}%]")
axes[0].axvline(ci_1d[1], color="black", linestyle=":")
axes[0].set_title("Bootstrap Difference: 1-Day Retention\n(Gate 40 - Gate 30)")
axes[0].set_xlabel("Difference (Percentage Points)")
axes[0].legend()

# Plot 7-Day Retention Diff
sns.histplot(boot_diff_7d, kde=True, ax=axes[1], color="#DD8452", stat="density", bins=40)
axes[1].axvline(0, color="red", linestyle="--", linewidth=1.5, label="No Difference (0%)")
axes[1].axvline(ci_7d[0], color="black", linestyle=":", label=f"95% CI: [{ci_7d[0]:.2f}%, {ci_7d[1]:.2f}%]")
axes[1].axvline(ci_7d[1], color="black", linestyle=":")
axes[1].set_title("Bootstrap Difference: 7-Day Retention\n(Gate 40 - Gate 30)")
axes[1].set_xlabel("Difference (Percentage Points)")
axes[1].legend()

plt.tight_layout()
out_fig_path = os.path.join(OUTPUT_DIR, 'ab_test_bootstrap_distributions.png')
plt.savefig(out_fig_path, bbox_inches='tight')
print(f"Saved bootstrap visualization to: {out_fig_path}")
plt.close()

## 9. Multiple Testing Risk & Correction
Because we evaluated 3 distinct endpoints (`retention_1`, `retention_7`, `sum_gamerounds`), the family-wise error rate (FWER) inflates to $1 - (1 - 0.05)^3 \approx 14.3\%$.  
Applying Bonferroni correction: $\alpha_{\text{Bonferroni}} = \frac{0.05}{3} \approx 0.0167$.

In [ ]:
p_vals = {'retention_1': 0.0739, 'retention_7': 0.00159, 'sum_gamerounds': 0.0509}
alpha_bonf = 0.05 / len(p_vals)

print(f"Bonferroni-adjusted significance threshold: alpha = {alpha_bonf:.4f}")
for m, p in p_vals.items():
    sig = "SIGNIFICANT" if p < alpha_bonf else "NOT SIGNIFICANT"
    print(f" - {m:<15}: p-value = {p:.5f} -> {sig}")

## 10. Practical Significance & Business Impact

* **Statistical Conclusion:** The 7-day retention decline ($-0.818$ percentage points, $p = 0.00159$) is robust to multiple testing corrections and non-parametric bootstrap validation (99.94% probability of decline).
* **Commercial Significance:** In mobile free-to-play gaming, long-term retention is the single strongest determinant of lifetime value (LTV) and advertising revenue:
  * A relative drop of **-4.30% in 7-day retention** represents roughly 8,200 lost weekly active players for every 1,000,000 installs.
  * Because `sum_gamerounds` showed **zero meaningful improvement** (mean: 51.34 vs. 51.30, $p = 0.95$), delaying the gate provides **zero upside** to offset this retention penalty.
* **Psychological / Behavioral Mechanism:**  
  *When players hit a gate at Level 30, they are forced to take a breather before hedonic adaptation/boredom sets in, prolonging their enjoyment and anticipation. Moving the gate to Level 40 causes more players to burn out on repetitive levels and uninstall the app.*
